In [ ]:
import wavefunction as wf
import grids as gr
import plot
import numpy as np

# Dirac 波函数

In [ ]:
# Dirac 波函数
# 1.构造DiracHydrogen实例
n, k, m, Z = 2, -2, -0.5, 1
Psi_dirac = wf.DiracHydrogen(n, k, m, Z)

In [ ]:
# 2.生成网格
grid = gr.GridGenerator(10, 50)
X, Y, Z = grid.generate_grid()

# 3.计算波函数
spinor = Psi_dirac.compute_psi_xyz(X, Y, Z, t=0)
_ = spinor.component_fractions()

In [ ]:
# 4.可视化波函数
plt_config = plot.DiracPlotConfig()

fig = plot.Dirac_plot(spinor, grid, plt_config)
fig.show()

# Schrodinger 波函数

In [ ]:
# Schrodinger 波函数
# 1.构造Hydrogen实例
n, l, m, Z = 2, 1, 1, 1
Psi = wf.SchrodingerHydrogen(n, l, m, Z)

In [ ]:
# 2.生成网格（可以用上面已有的）
grid = gr.GridGenerator(10, 50)
X, Y, Z = grid.generate_grid()

# 3.计算波函数
psi = Psi.compute_psi_xyz(X, Y, Z, t=0)

In [ ]:
# 4.可视化波函数
plt_config = plot.SchrodingerPlotConfig(xyzrange=10, color=('#66ccff', 'hsv'))

fig = plot.Schrodinger_plot(psi, grid, plt_config)
fig.show()

In [ ]:
# 实轨道
psi = Psi.compute_psi_xyz(X, Y, Z, t=0, isreal=True)
fig = plot.Schrodinger_plot(psi, grid, plt_config)
fig.show()

# 轨道混合

In [ ]:
# 轨道混合
# 生成网格
grid = gr.GridGenerator(10, 50)
X, Y, Z = grid.generate_grid()

p2 = wf.SchrodingerHydrogen(2, 1, 1, 1)
s2 = wf.SchrodingerHydrogen(2, 0, 0, 1)

# 3.计算波函数
psi_p2 = p2.compute_psi_xyz(X, Y, Z, t=0, isreal=True)
psi_s2 = s2.compute_psi_xyz(X, Y, Z, t=0)

# normalize设为True，将波函数归一化
# 将n个波函数按照第二个参数的比例混合
psi = wf.generate_hybrid_orbital([psi_p2, psi_s2], [1, 1], normalize=True)

# 4.可视化波函数
plt_config = plot.SchrodingerPlotConfig()

fig = plot.Schrodinger_plot(psi, grid, plt_config)
fig.show()

In [ ]:
# 复系数
psi = wf.generate_hybrid_orbital([psi_p2, psi_s2], [1, 1j], normalize=True)

# 4.可视化波函数
plt_config = plot.SchrodingerPlotConfig()

fig = plot.Schrodinger_plot(psi, grid, plt_config)
fig.show()

In [ ]:
# Dirac波函数
p2 = wf.DiracHydrogen(2, -2, -0.5, 1)
s2 = wf.DiracHydrogen(2, -1, -0.5, 1)

psi_p2 = p2.compute_psi_xyz(X, Y, Z, t=0)
psi_s2 = s2.compute_psi_xyz(X, Y, Z, t=0)

psi = wf.generate_hybrid_orbital([psi_p2, psi_s2], [1, 1])
# 返回BaseSpinor，需要转换为DiracSpinor
psi = wf.DiracSpinor(psi.psi)

plt_config = plot.DiracPlotConfig()

fig = plot.Dirac_plot(psi, grid, plt_config)
fig.show()

# LCAO-MO

In [ ]:
class H2MolecularOrbital:
    def __init__(self, n, l, m, Z, R=4):
        """
        H2分子轨道，通过LCAO方法组合两个氢原子波函数
        Parameters:
        -----------
        n, l, m : int, 主量子数、角量子数、磁量子数
        Z : int, 原子序数
        R : float, 两个氢原子核间距
        a_mu : float, 约化波尔半径
        mu : float, 约化质量
        """
        self.R = R
        self.atom1 = wf.SchrodingerHydrogen(n, l, m, Z)
        self.atom2 = wf.SchrodingerHydrogen(n, l, m, Z)
        self.c1 = 1 / np.sqrt(2)
        self.c2 = - 1 / np.sqrt(2)

    def compute_molecular_psi_xyz(self, x, y, z, t=0, isreal=False):
        """
        计算H2分子的波函数（LCAO）
        Parameters:
        -----------
        x, y, z : float or array, 笛卡尔坐标
        t : float, 时间
        isreal : bool, 是否返回实数波函数
        Returns:
        --------
        array-like : 分子波函数（复数数组）
        """
        psi1 = self.atom1.compute_psi_xyz(x, y, z + self.R / 2, t, isreal)
        psi2 = self.atom2.compute_psi_xyz(x, y, z - self.R / 2, t, isreal)
        psi_mol = self.c1 * psi1 + self.c2 * psi2
        return psi_mol


# 1. 构造H2分子实例
n, l, m, Z = 2, 1, 1, 1
R = 4
Psi_mol = H2MolecularOrbital(n, l, m, Z, R=R)

# 2. 生成网格
grid = gr.GridGenerator(10, 50)
X, Y, Z = grid.generate_grid()

# 3. 计算分子波函数
psi_mol = Psi_mol.compute_molecular_psi_xyz(X, Y, Z, t=0, isreal=True)

# 4. 可视化分子波函数
plt_config = plot.SchrodingerPlotConfig(
    plot_type='psi',
    xyzrange=10,
    color=('#66ccff', 'hsv'),
    width=1000,
    height=1000,
    psi_scale=0.2,
    camera=dict(
        eye=dict(x=2, y=1.5, z=0),
        up=dict(x=0, y=1, z=0),
        center=dict(x=0, y=0, z=0)))
fig = plot.Schrodinger_plot(psi_mol, grid, plt_config)
fig.show()